# 📊 FFT 주파수 분석 (Fast Fourier Transform)

> **학습 목표**: FFT를 사용하여 진동 신호의 주파수 특성을 분석합니다.

---

## 📋 학습 내용

1. ✅ FFT (Fast Fourier Transform) 개념
2. ✅ 시계열 데이터 → 주파수 도메인 변환
3. ✅ 주파수 스펙트럼 시각화
4. ✅ 지배 주파수 (Dominant Frequency) 찾기
5. ✅ 이상 주파수 탐지
6. ✅ Spectrogram (시간-주파수 분석)

**소요 시간**: 약 35분  
**난이도**: ⭐⭐⭐ (중상급)  
**사전 지식**: 시계열 분석 기초, NumPy

---

## 🔧 Step 1: 라이브러리 Import

In [ ]:
# 데이터 분석
import pandas as pd
import numpy as np

# 신호 처리
from scipy.fft import fft, fftfreq, rfft, rfftfreq
from scipy.signal import find_peaks, spectrogram, welch

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# 유틸리티
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ 라이브러리 로드 완료!")

## 📂 Step 2: 진동 데이터 로드

In [ ]:
# 데이터 파일 경로
data_path = Path('../data/sample_kamp_vibration.csv')

# 데이터 로드
def load_data(file_path):
    encodings = ['utf-8-sig', 'cp949', 'utf-8']
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"✅ 데이터 로드 성공! (인코딩: {encoding})")
            return df
        except:
            continue
    raise ValueError("데이터 로드 실패")

df = load_data(data_path)
print(f"\n📊 데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"💾 메모리 사용량: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# 데이터 미리보기
df.head()

In [ ]:
# 진동 관련 컬럼 찾기
vibration_cols = [col for col in df.columns if any(keyword in col.lower() 
                  for keyword in ['vib', 'x', 'y', 'z', '진동', 'accel'])]

print(f"📊 진동 관련 컬럼: {len(vibration_cols)}개")
print(vibration_cols)

# 첫 번째 진동 컬럼 선택
if len(vibration_cols) == 0:
    # 진동 컬럼이 없으면 첫 번째 수치형 컬럼 사용
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    vibration_col = numeric_cols[0] if numeric_cols else df.columns[0]
    print(f"\n⚠️ 진동 컬럼 미발견, 첫 번째 컬럼 사용: {vibration_col}")
else:
    vibration_col = vibration_cols[0]
    print(f"\n✅ 선택된 진동 컬럼: {vibration_col}")

# 진동 신호 추출
signal = df[vibration_col].dropna().values
print(f"\n📊 신호 정보:")
print(f"   샘플 수: {len(signal):,}개")
print(f"   값 범위: [{signal.min():.4f}, {signal.max():.4f}]")
print(f"   평균: {signal.mean():.4f}")
print(f"   표준편차: {signal.std():.4f}")

## 📈 Step 3: 시계열 신호 시각화

In [ ]:
# 샘플링 레이트 추정 (또는 사용자 지정)
sampling_rate = 1000  # Hz (1초에 1000개 샘플)
duration = len(signal) / sampling_rate

print(f"⚙️ 샘플링 설정:")
print(f"   샘플링 레이트: {sampling_rate} Hz")
print(f"   총 시간: {duration:.2f}초")
print(f"   시간 간격: {1/sampling_rate*1000:.2f}ms")

# 시간 축 생성
time = np.arange(len(signal)) / sampling_rate

# 시계열 플롯 (처음 1000개 샘플)
sample_size = min(1000, len(signal))

plt.figure(figsize=(14, 6))
plt.plot(time[:sample_size], signal[:sample_size], linewidth=0.8)
plt.title(f'시계열 진동 신호 (처음 {sample_size/sampling_rate:.2f}초)', fontsize=14, fontweight='bold')
plt.xlabel('시간 (초)')
plt.ylabel('진폭')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 시계열 도메인 (Time Domain):")
print(f"   - X축: 시간")
print(f"   - Y축: 진폭")
print(f"   - 패턴: 진동의 시간적 변화 관찰")

## 🔄 Step 4: FFT 변환 (주파수 도메인)

In [ ]:
# FFT 수행 (Real FFT - 실수 입력에 최적화)
print("🔄 FFT 변환 수행 중...")

# 신호 길이 (FFT 효율을 위해 2의 거듭제곱으로 조정 가능)
n_samples = len(signal)
n_fft = 2 ** int(np.ceil(np.log2(n_samples)))  # 다음 2의 거듭제곱

print(f"   원본 샘플 수: {n_samples:,}")
print(f"   FFT 샘플 수: {n_fft:,} (2^{int(np.log2(n_fft))})")

# FFT 계산
fft_values = rfft(signal, n=n_fft)
frequencies = rfftfreq(n_fft, 1/sampling_rate)

# 크기 (Magnitude) 계산
magnitudes = np.abs(fft_values)

# 정규화 (옵션)
magnitudes_normalized = magnitudes / n_samples

print(f"\n✅ FFT 완료!")
print(f"   주파수 개수: {len(frequencies):,}")
print(f"   주파수 범위: [0, {frequencies.max():.2f}] Hz")
print(f"   주파수 해상도: {frequencies[1]:.4f} Hz")

In [ ]:
# FFT 스펙트럼 시각화
plt.figure(figsize=(14, 6))
plt.plot(frequencies, magnitudes_normalized, linewidth=0.8)
plt.title('주파수 스펙트럼 (Frequency Spectrum)', fontsize=14, fontweight='bold')
plt.xlabel('주파수 (Hz)')
plt.ylabel('크기 (Magnitude)')
plt.grid(alpha=0.3)
plt.xlim(0, sampling_rate / 2)  # Nyquist 주파수까지만
plt.tight_layout()
plt.show()

print("\n💡 주파수 도메인 (Frequency Domain):")
print("   - X축: 주파수 (Hz)")
print("   - Y축: 크기 (해당 주파수 성분의 강도)")
print("   - 패턴: 주파수별 에너지 분포")
print(f"   - Nyquist 주파수: {sampling_rate/2} Hz (샘플링 레이트의 절반)")

## 🎯 Step 5: 지배 주파수 (Dominant Frequency) 찾기

In [ ]:
# Peak 검출
# 최소 높이 설정 (평균의 2배)
threshold = magnitudes_normalized.mean() * 2
peaks, properties = find_peaks(magnitudes_normalized, height=threshold, distance=10)

# 상위 10개 주파수 추출
top_n = min(10, len(peaks))
top_peak_indices = peaks[np.argsort(magnitudes_normalized[peaks])[-top_n:][::-1]]

print(f"🎯 지배 주파수 (상위 {top_n}개):")
print(f"\n{'순위':<5} {'주파수 (Hz)':<15} {'크기':<15} {'설명'}")
print("-" * 60)

for rank, peak_idx in enumerate(top_peak_indices, 1):
    freq = frequencies[peak_idx]
    mag = magnitudes_normalized[peak_idx]
    
    # 주파수 대역 분류
    if freq < 10:
        desc = "저주파 (회전 불균형)"
    elif freq < 100:
        desc = "중주파 (베어링 결함)"
    elif freq < 1000:
        desc = "고주파 (기어 결함)"
    else:
        desc = "초고주파 (공진)"
    
    print(f"{rank:<5} {freq:<15.2f} {mag:<15.6f} {desc}")

# 가장 지배적인 주파수
dominant_freq = frequencies[top_peak_indices[0]]
dominant_mag = magnitudes_normalized[top_peak_indices[0]]

print(f"\n⭐ 가장 지배적인 주파수: {dominant_freq:.2f} Hz (크기: {dominant_mag:.6f})")

In [ ]:
# Peak 표시된 FFT 스펙트럼
plt.figure(figsize=(14, 6))
plt.plot(frequencies, magnitudes_normalized, linewidth=0.8, label='FFT Spectrum')
plt.plot(frequencies[top_peak_indices], magnitudes_normalized[top_peak_indices], 
         'ro', markersize=8, label=f'Top {top_n} Peaks')

# 지배 주파수에 레이블 추가
for peak_idx in top_peak_indices[:5]:  # 상위 5개만
    freq = frequencies[peak_idx]
    mag = magnitudes_normalized[peak_idx]
    plt.annotate(f'{freq:.1f} Hz', 
                xy=(freq, mag), 
                xytext=(10, 10), 
                textcoords='offset points',
                fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

plt.title('주파수 스펙트럼 with Peak Detection', fontsize=14, fontweight='bold')
plt.xlabel('주파수 (Hz)')
plt.ylabel('크기')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(0, sampling_rate / 2)
plt.tight_layout()
plt.show()

## 🚨 Step 6: 이상 주파수 탐지

In [ ]:
# 이상 주파수 기준 설정
# 방법 1: 통계적 임계값 (평균 + 3*표준편차)
mean_mag = magnitudes_normalized.mean()
std_mag = magnitudes_normalized.std()
anomaly_threshold = mean_mag + 3 * std_mag

# 이상치 검출
anomaly_indices = np.where(magnitudes_normalized > anomaly_threshold)[0]
anomaly_freqs = frequencies[anomaly_indices]
anomaly_mags = magnitudes_normalized[anomaly_indices]

print(f"🚨 이상 주파수 탐지 결과:")
print(f"   임계값: {anomaly_threshold:.6f} (평균 + 3σ)")
print(f"   이상 주파수 개수: {len(anomaly_freqs)}개")

if len(anomaly_freqs) > 0:
    print(f"\n📊 이상 주파수 목록 (상위 10개):")
    top_anomalies = min(10, len(anomaly_freqs))
    sorted_indices = np.argsort(anomaly_mags)[-top_anomalies:][::-1]
    
    print(f"\n{'순위':<5} {'주파수 (Hz)':<15} {'크기':<15} {'위험도'}")
    print("-" * 60)
    
    for rank, idx in enumerate(sorted_indices, 1):
        freq = anomaly_freqs[idx]
        mag = anomaly_mags[idx]
        risk = "🔴 높음" if mag > mean_mag + 5*std_mag else "🟡 중간"
        print(f"{rank:<5} {freq:<15.2f} {mag:<15.6f} {risk}")
else:
    print("\n✅ 이상 주파수 없음")

In [ ]:
# 이상 주파수 시각화
plt.figure(figsize=(14, 6))
plt.plot(frequencies, magnitudes_normalized, linewidth=0.8, label='Normal', alpha=0.7)
plt.axhline(y=anomaly_threshold, color='r', linestyle='--', label=f'임계값 (평균 + 3σ)', linewidth=2)

if len(anomaly_freqs) > 0:
    plt.scatter(anomaly_freqs, anomaly_mags, color='red', s=50, 
               label=f'이상 주파수 ({len(anomaly_freqs)}개)', zorder=5)

plt.title('이상 주파수 탐지 (Anomaly Frequency Detection)', fontsize=14, fontweight='bold')
plt.xlabel('주파수 (Hz)')
plt.ylabel('크기')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(0, sampling_rate / 2)
plt.tight_layout()
plt.show()

print("\n💡 이상 주파수 의미:")
print("   - 예상치 못한 고주파: 공진, 느슨한 부품")
print("   - 비정상적 저주파: 회전 불균형, 미스얼라인먼트")
print("   - 특정 주파수 급증: 베어링 결함, 기어 마모")

## 📊 Step 7: Spectrogram (시간-주파수 분석)

In [ ]:
# Spectrogram 계산
print("🔄 Spectrogram 계산 중...")

# 신호 길이 제한 (메모리 효율)
signal_sample = signal[:min(10000, len(signal))]

frequencies_spec, times_spec, Sxx = spectrogram(
    signal_sample, 
    fs=sampling_rate,
    nperseg=256,  # 윈도우 크기
    noverlap=128  # 오버랩
)

print(f"✅ Spectrogram 완료!")
print(f"   시간 분할: {len(times_spec)}개")
print(f"   주파수 분할: {len(frequencies_spec)}개")
print(f"   시간 범위: [0, {times_spec.max():.2f}]초")
print(f"   주파수 범위: [0, {frequencies_spec.max():.2f}]Hz")

In [ ]:
# Spectrogram 시각화
plt.figure(figsize=(14, 8))
plt.pcolormesh(times_spec, frequencies_spec, 10 * np.log10(Sxx + 1e-10), 
               shading='gouraud', cmap='viridis')
plt.colorbar(label='강도 (dB)')
plt.title('Spectrogram (시간-주파수 분석)', fontsize=14, fontweight='bold')
plt.xlabel('시간 (초)')
plt.ylabel('주파수 (Hz)')
plt.ylim(0, sampling_rate / 2)
plt.tight_layout()
plt.show()

print("\n💡 Spectrogram 해석:")
print("   - X축: 시간")
print("   - Y축: 주파수")
print("   - 색상: 에너지 강도 (밝을수록 강함)")
print("   - 패턴: 주파수가 시간에 따라 어떻게 변하는지 관찰")
print("   - 활용: 비정상 주파수 발생 시점 파악")

## 💾 Step 8: 결과 저장

In [ ]:
# 출력 디렉토리 생성
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# 1. 지배 주파수 저장
dominant_freqs_df = pd.DataFrame({
    'rank': range(1, len(top_peak_indices) + 1),
    'frequency_hz': frequencies[top_peak_indices],
    'magnitude': magnitudes_normalized[top_peak_indices]
})
dominant_file = output_dir / '02_dominant_frequencies.csv'
dominant_freqs_df.to_csv(dominant_file, index=False, encoding='utf-8-sig')
print(f"✅ 지배 주파수 저장: {dominant_file}")

# 2. 이상 주파수 저장
if len(anomaly_freqs) > 0:
    anomaly_df = pd.DataFrame({
        'frequency_hz': anomaly_freqs,
        'magnitude': anomaly_mags
    }).sort_values('magnitude', ascending=False)
    anomaly_file = output_dir / '02_anomaly_frequencies.csv'
    anomaly_df.to_csv(anomaly_file, index=False, encoding='utf-8-sig')
    print(f"✅ 이상 주파수 저장: {anomaly_file}")
else:
    print("⚠️ 이상 주파수 없음 (저장 생략)")

# 3. FFT 스펙트럼 저장 (NumPy)
fft_file = output_dir / '02_fft_spectrum.npz'
np.savez(fft_file, 
         frequencies=frequencies, 
         magnitudes=magnitudes_normalized,
         sampling_rate=sampling_rate)
print(f"✅ FFT 스펙트럼 저장: {fft_file}")

# 4. 분석 요약 저장
summary = pd.DataFrame({
    '항목': ['샘플링 레이트', '샘플 수', '총 시간', '주파수 해상도', 
            '지배 주파수', '이상 주파수 개수', '임계값'],
    '값': [
        f"{sampling_rate} Hz",
        f"{n_samples:,}개",
        f"{duration:.2f}초",
        f"{frequencies[1]:.4f} Hz",
        f"{dominant_freq:.2f} Hz",
        f"{len(anomaly_freqs)}개",
        f"{anomaly_threshold:.6f}"
    ]
})
summary_file = output_dir / '02_fft_analysis_summary.csv'
summary.to_csv(summary_file, index=False, encoding='utf-8-sig')
print(f"✅ 분석 요약 저장: {summary_file}")

print("\n🎉 FFT 주파수 분석 완료!")

---

## 🎯 학습 정리

### ✅ 완료한 내용
1. FFT로 시계열 진동 신호를 주파수 도메인으로 변환
2. 주파수 스펙트럼 시각화 및 해석
3. Peak Detection으로 지배 주파수 추출
4. 통계적 방법으로 이상 주파수 탐지
5. Spectrogram으로 시간-주파수 분석
6. 분석 결과 저장 및 문서화

### 💡 핵심 인사이트
- **FFT의 장점**:
  - 시계열에서 보이지 않던 주파수 패턴 발견
  - 특정 주파수 성분 분리 및 분석
  - 설비 결함 조기 진단 (베어링, 기어, 회전 불균형)

- **주파수 대역별 의미**:
  - **저주파 (<10Hz)**: 회전 불균형, 미스얼라인먼트
  - **중주파 (10-100Hz)**: 베어링 결함, 구조적 공진
  - **고주파 (100-1000Hz)**: 기어 결함, 벨트 문제
  - **초고주파 (>1000Hz)**: 공진, 느슨한 부품

- **이상 탐지 방법**:
  - 3-sigma 규칙: 평균 + 3×표준편차
  - 정상 운전 데이터와 비교
  - 시간에 따른 주파수 변화 추적 (Spectrogram)

- **실무 적용**:
  - 예지보전: 고장 전 징후 감지
  - 품질 관리: 진동 기준 초과 탐지
  - 근본 원인 분석: 특정 주파수로 결함 위치 파악

### 📚 다음 단계
- **03_anomaly_detection.ipynb**: 머신러닝 기반 이상 탐지
- **Part 2-2**: Vision Transformer (ViT) 활용

### 🔗 참고 자료
- [FFT 개념 이해](https://en.wikipedia.org/wiki/Fast_Fourier_transform)
- [진동 분석 가이드](https://www.azosensors.com/article.aspx?ArticleID=1467)
- [SciPy FFT 문서](https://docs.scipy.org/doc/scipy/tutorial/fft.html)

---

*제조AI 교육 v12 Enhanced | Part 2-1 | 2025.02*